# Jalon B3 - Compare Two Runs\n\nNotebook pour comparer deux runs B3 (metrics, top contributors, attribution map).

In [ ]:
from pathlib import Path\nimport pandas as pd\nfrom boardGPT.interpretability import load_run_artifacts

In [ ]:
root = Path('out_interp/jalon_b3')\nruns = sorted([p for p in root.glob('*') if p.is_dir()])\nif len(runs) < 2:\n    raise RuntimeError('Il faut au moins 2 runs B3. Lance: python tools/interpretability_cli.py smoke')\nrun_a, run_b = runs[-2], runs[-1]\nrun_a, run_b

In [ ]:
a = load_run_artifacts(str(run_a))\nb = load_run_artifacts(str(run_b))\n\nprint('Run A:', run_a.name)\nprint('Run B:', run_b.name)

In [ ]:
df_a = pd.read_csv(run_a / 'metrics_by_layer.csv')\ndf_b = pd.read_csv(run_b / 'metrics_by_layer.csv')\n\ndf_compare = df_a.merge(df_b, on='layer', suffixes=('_a', '_b'))\ndf_compare['success_rate_delta'] = df_compare['success_rate_b'] - df_compare['success_rate_a']\ndf_compare['target_gain_mean_delta'] = df_compare['target_gain_mean_b'] - df_compare['target_gain_mean_a']\ndf_compare

In [ ]:
top_a = pd.read_csv(run_a / 'top_contributors.csv')\ntop_b = pd.read_csv(run_b / 'top_contributors.csv')\n\nprint('Top A')\ndisplay(top_a.head(10))\nprint('Top B')\ndisplay(top_b.head(10))

In [ ]:
amap_a = a['tensors'].get('attribution_map')\namap_b = b['tensors'].get('attribution_map')\n\nif amap_a is None or amap_b is None:\n    print('attribution_map absent dans un des runs')\nelse:\n    print('shape A:', tuple(amap_a.shape))\n    print('shape B:', tuple(amap_b.shape))\n    print('mean abs diff:', (amap_b - amap_a).abs().mean().item())